# Step-by-Step: Run Backend or Full Stack on Free Google Colab

This notebook can run either:
- backend only on Colab
- backend + frontend together on Colab

Important sync rule:
- Colab only sees code that is pushed to GitHub or uploaded into the Colab runtime.
- If you changed files in VS Code, commit and push those changes first, then rerun the clone or one-click cell here.
- Backend dependencies are installed from `multiagent/requirement.txt`.
- Frontend dependencies are installed from `multiagent/package.json`.

Backend flow in this notebook:
- Step 1: Clone the GitHub repository
- Step 2: Install Linux + Python dependencies
- Step 3: Start FastAPI backend on port 8000
- Step 4: Create temporary public URL using Cloudflare tunnel
- Step 5: Verify API health endpoint

Notes:
- No upload step is needed for the GitHub workflow.
- No Google Drive mount is needed.
- Tunnel URLs are temporary and change each session.

In [ ]:
# One-click backend run: refresh repo, install current backend deps, start backend, tunnel it, verify
import os
import re
import subprocess
import sys
import time

import requests

REPO_URL = "https://github.com/DinethiWijesinghe/multiagent.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/multiagent"
CLOUDFLARED = "/content/cloudflared-linux-amd64"
RUNTIME_PROFILE = "LITE"
RAG_LLM_PROVIDER = "none"

requirements_path = os.path.join(REPO_DIR, "multiagent", "requirement.txt")

print("[1/5] Refreshing repository from GitHub...")
subprocess.run(["rm", "-rf", REPO_DIR], check=False)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

if not os.path.exists(requirements_path):
    raise FileNotFoundError(f"Missing backend requirements file: {requirements_path}")

print("[2/5] Installing Linux packages and backend dependencies...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr", "libgl1", "libglib2.0-0", "wget", "git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", requirements_path], check=True)

print("[3/5] Starting FastAPI backend...")
backend_env = os.environ.copy()
backend_env["RUNTIME_PROFILE"] = RUNTIME_PROFILE
backend_env["RAG_LLM_PROVIDER"] = RAG_LLM_PROVIDER
backend_env["PYTHONUNBUFFERED"] = "1"
server_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "multiagent.api_server:app",
        "--host", "0.0.0.0",
        "--port", "8000",
    ],
    cwd=REPO_DIR,
    env=backend_env,
)

local_health = None
last_error = None
for attempt in range(1, 21):
    try:
        local_health = requests.get("http://127.0.0.1:8000/health", timeout=20)
        if local_health.status_code == 200:
            break
    except Exception as exc:
        last_error = exc
    print(f"Waiting for backend {attempt}/20...")
    time.sleep(3)
else:
    raise RuntimeError(f"Backend did not become healthy: {last_error}")

print("Local health:", local_health.status_code)
print(local_health.json())
print(f"Runtime profile: {RUNTIME_PROFILE}")
print(f"RAG provider: {RAG_LLM_PROVIDER}")

print("[4/5] Creating Cloudflare tunnel...")
subprocess.run(["rm", "-rf", CLOUDFLARED], check=False)
subprocess.run(["curl", "-fL", "--retry", "5", "--retry-delay", "2", "--retry-all-errors", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", CLOUDFLARED], check=True)
subprocess.run(["chmod", "+x", CLOUDFLARED], check=True)

tunnel_process = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd="/content",
)

public_url = None
start = time.time()
while time.time() - start < 90:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    print(line.rstrip())
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0).strip()
        break

if not public_url:
    raise RuntimeError("Cloudflare tunnel URL was not created. Re-run this cell.")

print("[5/5] Verifying public health endpoint...")
test_url = f"{public_url}/health"
last_error = None
for attempt in range(1, 9):
    try:
        public_health = requests.get(test_url, timeout=25)
        print("Public health:", public_health.status_code)
        print(public_health.json())
        break
    except Exception as exc:
        last_error = exc
        print(f"Attempt {attempt}/8 failed: {exc}")
        time.sleep(5)
else:
    raise RuntimeError(f"Tunnel created but public health failed: {last_error}")

print("\nDone.")
print("Backend public URL:", public_url)
print("If you want your laptop frontend to call Colab, set this in multiagent/.env:")
print(f"VITE_API_URL={public_url}")

# One-Click Run

If you want to avoid running Step 1 to Step 5 one by one, run the next code cell once.

What it does automatically:
- clones the latest GitHub code
- installs Linux and Python dependencies
- starts the FastAPI backend
- creates a Cloudflare tunnel
- verifies both local and public health endpoints

Use the manual Step 1 to Step 5 cells only if you want to debug one stage at a time.

In [ ]:
# Step 1: Clone latest repository from GitHub
REPO_URL = "https://github.com/DinethiWijesinghe/multiagent.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/multiagent"

!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Step 2: Install Dependencies

This installs OCR system libraries and Python packages needed by the FastAPI server.

In [ ]:
# Step 2 code: Linux + Python dependencies from the current repo
import os
import sys

%cd /content/multiagent

requirements_path = "/content/multiagent/multiagent/requirement.txt"
if not os.path.exists(requirements_path):
    raise FileNotFoundError(f"Missing backend requirements file: {requirements_path}")

!apt-get update -qq
!apt-get install -y -qq tesseract-ocr libgl1 libglib2.0-0 wget git

!python -m pip install --upgrade pip setuptools wheel
!python -m pip install -r {requirements_path}

print("Installed backend dependencies from:", requirements_path)

# Step 3: Start FastAPI Server

This starts backend at http://127.0.0.1:8000 inside Colab.

In [ ]:
# Step 3 code: run backend with the current runtime defaults and test local health
import os
import subprocess
import time
import requests

%cd /content/multiagent

backend_env = os.environ.copy()
backend_env["RUNTIME_PROFILE"] = "LITE"
backend_env["RAG_LLM_PROVIDER"] = "none"
backend_env["PYTHONUNBUFFERED"] = "1"

server_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "multiagent.api_server:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd="/content/multiagent",
    env=backend_env,
    )

time.sleep(8)

try:
    health = requests.get("http://127.0.0.1:8000/health", timeout=15)
    print("Local health:", health.status_code)
    print(health.json())
    print("Runtime profile:", backend_env["RUNTIME_PROFILE"])
    print("RAG provider:", backend_env["RAG_LLM_PROVIDER"])
except Exception as exc:
    print("Health check failed:", exc)

# Step 4: Create Public URL (Cloudflare Tunnel)

This gives a temporary internet URL to your Colab backend.

In [ ]:
# Step 4 code: create Cloudflare public URL
import re
import subprocess
import time

!rm -rf cloudflared-linux-amd64
!curl -fL --retry 5 --retry-delay 2 --retry-all-errors https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# Stop previous tunnel process in this runtime if present
try:
    tunnel_process.poll()
except Exception:
    pass

tunnel_process = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
start = time.time()

while time.time() - start < 90:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    print(line.rstrip())
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0).strip()
        break

if public_url:
    print("\nPublic URL:", public_url)
    print("Now run Step 5 to verify it.")
else:
    print("\nPublic URL not found. Re-run this step.")

# Step 5: Verify Public API

Run this after Step 4 to confirm the public URL works.

In [ ]:
# Step 5 code: verify public health endpoint with retries
import requests
import time

if not public_url:
    print("No public URL detected. Re-run Step 4.")
else:
    test_url = f"{public_url.rstrip('/')}/health"
    print("Testing:", test_url)

    ok = False
    last_err = None

    # DNS for trycloudflare can take a short time to propagate
    for attempt in range(1, 9):
        try:
            response = requests.get(test_url, timeout=25)
            print("Public health:", response.status_code)
            print(response.json())
            ok = True
            break
        except Exception as e:
            last_err = e
            print(f"Attempt {attempt}/8 failed: {e}")
            time.sleep(5)

    if not ok:
        print("\nPublic URL could not be reached yet.")
        print("Action: Re-run Step 4 to generate a fresh URL, then run Step 5 again.")
        print("Last error:", last_err)

# Full Stack in Colab (Backend + Frontend)

Run the next single cell to:
- refresh the repo from GitHub so pushed VS Code changes are included
- install backend dependencies from `multiagent/requirement.txt`
- install frontend dependencies from `multiagent/package.json`
- start FastAPI backend on port 8000 with `RUNTIME_PROFILE=LITE` and `RAG_LLM_PROVIDER=none`
- create a backend public tunnel URL
- start Vite frontend on port 5173 with `VITE_API_URL` set to the backend tunnel
- create a frontend public tunnel URL

This gives you two live links:
- Frontend URL: open this in your browser
- Backend URL: already wired into the frontend

Notes:
- Push your latest VS Code changes to GitHub before running this cell if you want Colab to use them.
- Keep the Colab runtime running while you use the app.
- Tunnel URLs are temporary and change when the runtime restarts.

In [ ]:
# One-click full stack: refresh repo, install current deps, start backend + frontend, create two public URLs
import os
import re
import subprocess
import sys
import time

import requests

REPO_URL = "https://github.com/DinethiWijesinghe/multiagent.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/multiagent"
FRONTEND_DIR = f"{REPO_DIR}/multiagent"
REQUIREMENTS_PATH = f"{FRONTEND_DIR}/requirement.txt"
CLOUDFLARED = "/content/cloudflared-linux-amd64"
RUNTIME_PROFILE = "LITE"
RAG_LLM_PROVIDER = "none"


def run(cmd, cwd=None, shell=False):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd, shell=shell, executable="/bin/bash" if shell else None)


def wait_http_ok(url, attempts=20, delay=2, expect_status=None):
    last_error = None
    for i in range(1, attempts + 1):
        try:
            response = requests.get(url, timeout=20)
            if expect_status is not None and response.status_code == expect_status:
                return response
            if expect_status is None and response.status_code < 500:
                return response
        except Exception as exc:
            last_error = exc
        print(f"Wait {i}/{attempts}: {url} not ready yet")
        time.sleep(delay)
    raise RuntimeError(f"Failed to reach {url}. Last error: {last_error}")


def start_tunnel(local_url):
    proc = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", local_url, "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd="/content",
    )

    public_url = None
    start = time.time()
    while time.time() - start < 120:
        line = proc.stdout.readline()
        if not line:
            continue
        print(line.rstrip())
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0).strip()
            break

    if not public_url:
        raise RuntimeError(f"Could not create Cloudflare URL for {local_url}")

    return proc, public_url


print("[1/8] Refreshing repository from GitHub...")
run(["rm", "-rf", REPO_DIR])
run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])

if not os.path.exists(REQUIREMENTS_PATH):
    raise FileNotFoundError(f"Missing backend requirements file: {REQUIREMENTS_PATH}")

print("[2/8] Installing system dependencies...")
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "tesseract-ocr", "libgl1", "libglib2.0-0", "wget", "curl", "ca-certificates", "gnupg", "git"])

print("[3/8] Installing backend Python dependencies from multiagent/requirement.txt...")
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-r", REQUIREMENTS_PATH])

print("[4/8] Installing Node.js 20 for the Vite frontend...")
run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -", shell=True)
run(["apt-get", "install", "-y", "-qq", "nodejs"])
run(["node", "--version"])
run(["npm", "--version"])

print("[5/8] Installing frontend dependencies from package.json...")
run(["npm", "install"], cwd=FRONTEND_DIR)

print("[6/8] Starting backend and creating backend public URL...")
backend_env = os.environ.copy()
backend_env["RUNTIME_PROFILE"] = RUNTIME_PROFILE
backend_env["RAG_LLM_PROVIDER"] = RAG_LLM_PROVIDER
backend_env["PYTHONUNBUFFERED"] = "1"
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "multiagent.api_server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=REPO_DIR,
    env=backend_env,
    )
wait_http_ok("http://127.0.0.1:8000/health", attempts=25, delay=3, expect_status=200)
run(["rm", "-rf", CLOUDFLARED])
run(["curl", "-fL", "--retry", "5", "--retry-delay", "2", "--retry-all-errors", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", CLOUDFLARED])
run(["chmod", "+x", CLOUDFLARED])
backend_tunnel_proc, backend_public_url = start_tunnel("http://127.0.0.1:8000")
wait_http_ok(f"{backend_public_url}/health", attempts=20, delay=3, expect_status=200)
print("Backend public URL:", backend_public_url)

print("[7/8] Starting frontend with the backend tunnel URL injected...")
frontend_env = os.environ.copy()
frontend_env["VITE_API_URL"] = backend_public_url
frontend_proc = subprocess.Popen(
    ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173", "--strictPort", "--allowed-hosts", "all"],
    cwd=FRONTEND_DIR,
    env=frontend_env,
    )
wait_http_ok("http://127.0.0.1:5173", attempts=35, delay=2)

print("[8/8] Creating frontend public URL and verifying it...")
frontend_tunnel_proc, frontend_public_url = start_tunnel("http://127.0.0.1:5173")
wait_http_ok(frontend_public_url, attempts=20, delay=3)

print("\n=== FULL STACK LIVE ON COLAB ===")
print("Frontend URL (open this):", frontend_public_url)
print("Backend URL:", backend_public_url)
print("Runtime profile:", RUNTIME_PROFILE)
print("RAG provider:", RAG_LLM_PROVIDER)
print("\nIf you edit code in VS Code later, push those changes to GitHub and rerun this full-stack cell to refresh Colab.")

# Tunnel Health Check and Expiry Reminder

Run the next cell any time to quickly verify whether your current backend and frontend tunnel URLs are still alive.

If a check fails, rerun the **Full Stack in Colab (Backend + Frontend)** one-click cell to generate fresh URLs.

In [ ]:
# Quick tunnel health check: backend + frontend
import requests
from urllib.parse import urlparse

backend_url = globals().get("backend_public_url") or globals().get("public_url")
frontend_url = globals().get("frontend_public_url")


def _normalize_base(url):
    if not url:
        return None
    return str(url).rstrip("/")


def _request_ok(url, timeout=15):
    try:
        r = requests.get(url, timeout=timeout)
        return True, r.status_code, None
    except Exception as exc:
        return False, None, str(exc)


def _check_backend(url):
    base = _normalize_base(url)
    if not base:
        print("Backend URL not found in current runtime variables.")
        return False

    health_url = f"{base}/health"
    ok, status, err = _request_ok(health_url)
    if ok and status == 200:
        print(f"Backend OK: {health_url} (status {status})")
        return True

    print(f"Backend FAILED: {health_url}")
    if status is not None:
        print(f"Status: {status}")
    if err:
        print(f"Error: {err}")
    return False


def _check_frontend(url):
    base = _normalize_base(url)
    if not base:
        print("Frontend URL not found in current runtime variables.")
        return False

    ok, status, err = _request_ok(base)
    if ok and status < 500:
        print(f"Frontend OK: {base} (status {status})")
        return True

    print(f"Frontend FAILED: {base}")
    if status is not None:
        print(f"Status: {status}")
    if err:
        print(f"Error: {err}")
    return False


print("Checking active tunnel URLs...\n")
backend_ok = _check_backend(backend_url)
frontend_ok = _check_frontend(frontend_url)

print("\nSummary")
print(f"Backend healthy: {backend_ok}")
print(f"Frontend healthy: {frontend_ok}")

if not (backend_ok and frontend_ok):
    print("\nReminder: Tunnel URLs are temporary.")
    print("Action: Rerun the Full Stack in Colab (Backend + Frontend) one-click cell to get fresh URLs.")
else:
    print("\nAll good. Your current tunnel URLs are active.")